# Meliora: worked examples

These 29 examples follow the method names and chapter order in *Statistical Tests for Credit Risk*. Each section introduces a method, calculates its result and explains how to read it. Small illustrative datasets make the calculations easy to follow.

Clone the [Meliora repository](https://github.com/at621/meliora), then open this notebook in your Python 3.11+ Jupyter environment with NumPy, pandas, SciPy and scikit-learn available. Run the setup cell below once to import Meliora directly from the checkout. pandas creates the input tables used by Meliora.

[Discrimination](#Discrimination) · [Calibration](#Calibration) · [Association](#Association) · [Stability](#Stability) · [LGD validation](#LGD-validation)

In [1]:
from pathlib import Path
import sys

import pandas as pd

# Locate the cloned package from the repository root or a notebook subfolder.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "meliora/core.py").is_file():
        sys.path.insert(0, str(candidate))
        break
import meliora as m

## Discrimination

### 1. ROC AUC

ROC AUC measures how well a score ranks defaulters above survivors. In this example, higher scores mean greater default risk; a tied pair receives half credit.

[Method reference and sources](../reference/roc_auc.md).

In [2]:
data = pd.DataFrame({'y': [0, 0, 1, 1], 'score': [1, 2, 2, 3]})
result = m.roc_auc(data, 'y', 'score')
print(f"ROC AUC: {result:.3f}")

ROC AUC: 0.875


**Interpretation.** Three positive-negative pairs are correctly ordered and one ties: (3+0.5)/4=0.875.

### 2. Gini coefficient

The Gini coefficient, also called the accuracy ratio, rescales ROC AUC as $2\,\mathrm{AUC}-1$. Perfect ranking gives 1 and random ranking gives 0.

[Method reference and sources](../reference/gini.md).

In [3]:
data = pd.DataFrame({'y': [0, 0, 1, 1], 'score': [1, 2, 2, 3]})
result = m.gini(data, 'y', 'score')
print(f"Gini coefficient: {result:.3f}")

Gini coefficient: 0.750


**Interpretation.** An AUC of 0.875 corresponds to Gini 0.75.

### 3. Kolmogorov-Smirnov statistic

The KS statistic is the largest gap between the score distributions of defaulters and survivors. The example also reports the two-sided test of equal distributions.

[Method reference and sources](../reference/kolmogorov_smirnov_stat.md).

In [4]:
data = pd.DataFrame({'y': [0, 0, 1, 1], 'score': [1, 2, 3, 4]})
result = m.kolmogorov_smirnov_stat(data, 'y', 'score')
print(f"KS statistic: {result.statistic:.3f}")
print(f"p-value: {result.pvalue:.3f}")

KS statistic: 1.000
p-value: 0.333


**Interpretation.** Class score ranges do not overlap (D=1), but the exact two-sided p-value is 1/3 for two observations per class.

### 4. Minimum empirical threshold error

This measure finds the score cut-off with the fewest classification mistakes in the sample. False alarms and missed defaults receive equal cost.

[Method reference and sources](../reference/bayesian_error_rate.md).

In [5]:
data = pd.DataFrame({'y': [0, 0, 1, 1], 'score': [1, 2, 2, 3]})
result = m.bayesian_error_rate(data, 'y', 'score')
print(f"Minimum classification error: {result:.1%}")

Minimum classification error: 25.0%


**Interpretation.** A positive and negative tie at score 2, so at least one of four observations is misclassified.

### 5. Information value

Information value compares the distribution of survivors and defaulters across bins. The output shows each bin's shares, weight of evidence and contribution to the total.

[Method reference and sources](../reference/information_value.md).

In [6]:
data = pd.DataFrame({'bin': ['A'] * 4 + ['B'] * 4, 'y': [0, 0, 0, 1, 0, 1, 1, 1]})
result = m.information_value(data, 'bin', 'y', smoothing=0)
details, information_value = result
print(f"Information value: {information_value:.3f}")
details.round(3)

Information value: 1.099


,good,bad,good_share,bad_share,WoE,IV
bin,,,,,,
A,3.0,1.0,0.75,0.25,1.099,0.549
B,1.0,3.0,0.25,0.75,-1.099,0.549


**Interpretation.** All cells are positive, allowing smoothing=0. The unsmoothed IV is log(3).

### 6. Conditional information entropy ratio

This ratio measures the share of uncertainty about default removed by knowing the grade. The example uses two equally sized grades: one with no defaults and one where every account defaults.

[Method reference and sources](../reference/conditional_information_entropy_ratio.md).

In [7]:
data = pd.DataFrame({'rate': [0, 1], 'n': [10, 10]})
result = m.conditional_information_entropy_ratio(data, 'rate', 'n')
print(f"Conditional information entropy ratio: {result:.3f}")

Conditional information entropy ratio: 1.000


**Interpretation.** Deterministic grade outcomes explain all marginal uncertainty in this balanced portfolio.

### 7. Mutual information

Mutual information measures how much knowing the grade reduces uncertainty about default, expressed in nats. The example uses the same two grades as the entropy-ratio example.

[Method reference and sources](../reference/kullback_leibler_dist.md).

In [8]:
data = pd.DataFrame({'rate': [0, 1], 'n': [10, 10]})
result = m.kullback_leibler_dist(data, 'rate', 'n')
print(f"Mutual information: {result:.3f} nats")

Mutual information: 0.693 nats


**Interpretation.** Perfect grade separation removes log(2) nats of uncertainty from a balanced binary outcome.

### 8. Cumulative LGD accuracy ratio

The cumulative LGD accuracy ratio summarises asymmetric agreement between predicted and realised loss grades across grade thresholds. Both use the same ordered scale, with higher grades representing larger losses.

[Method reference and sources](../reference/cumulative_lgd_accuracy_ratio.md).

In [9]:
data = pd.DataFrame({'p': [1, 2, 3, 3, 4], 'y': [1, 3, 2, 4, 4]})
result = m.cumulative_lgd_accuracy_ratio(data, 'p', 'y')
print(f"Cumulative LGD accuracy ratio: {result:.3f}")

Cumulative LGD accuracy ratio: 0.880


**Interpretation.** The threshold calculation gives CLAR = 0.880. As the book explains, this measure describes asymmetric agreement across loss grades; a high value alone does not establish good ranking.

### 9. Loss capture ratio

The loss capture ratio measures how quickly realised monetary losses accumulate when facilities are ordered by predicted LGD. It compares that ordering with the ideal ordering by realised LGD, using cumulative exposure on the horizontal axis.

[Method reference and sources](../reference/loss_capture_ratio.md).

In [10]:
result = m.loss_capture_ratio([1, 1, 1], [.1, .4, .9], [.1, .4, .9])
print(f"Loss capture ratio: {result:.3f}")

Loss capture ratio: 1.000


**Interpretation.** Sorting on true loss rates reproduces the ideal curve. Match the area convention when comparing implementations.

## Calibration

### 10. Binomial test

The binomial test assesses whether a grade's predicted PD is too low relative to its observed defaults. It uses independent default outcomes and a common PD within each grade; the example applies a 5% significance level.

[Method reference and sources](../reference/binomial_test.md).

In [11]:
data = pd.DataFrame({'grade': ['A'] * 4 + ['B'] * 4, 'outcome': [0, 0, 1, 1] * 2, 'pd': [.2] * 4 + [.6] * 4})
result = m.binomial_test(data, 'grade', 'outcome', 'pd')
result.round(3)

,Rating class,Predicted PD,Total count,Defaults,Actual Default Rate,p_value,Reject H0
0,A,0.2,4,2.0,0.5,0.181,False
1,B,0.6,4,2.0,0.5,0.821,False


**Interpretation.** Neither grade rejects at 5%. Four observations per grade give little power; non-rejection does not establish calibration.

### 11. Jeffreys test

The Jeffreys test combines observed defaults with a Beta(1/2, 1/2) prior for the grade default rate. A small posterior probability below the predicted PD indicates possible underestimation.

[Method reference and sources](../reference/jeffreys_test.md).

In [12]:
data = pd.DataFrame({'grade': ['A'] * 4, 'outcome': [0, 0, 1, 1], 'pd': [.5] * 4})
result = m.jeffreys_test(data, 'grade', 'outcome', 'pd')
result.round(3)

,Rating class,Predicted PD,Total count,Defaults,Actual Default Rate,p_value,Reject H0
0,A,0.5,4,2.0,0.5,0.5,False


**Interpretation.** The symmetric posterior has lower-tail probability 0.5 at the forecast PD.

### 12. Hosmer test

The Hosmer test jointly compares fixed grade PDs with observed default counts using a chi-square statistic. This example uses externally specified PDs and the default degrees of freedom.

[Method reference and sources](../reference/hosmer_test.md).

In [13]:
data = pd.DataFrame({'grade': ['A'] * 4 + ['B'] * 4, 'outcome': [0, 0, 1, 1] * 2, 'pd': [.2] * 4 + [.6] * 4})
result = m.hosmer_test(data, 'grade', 'outcome', 'pd')
p_value, reject = result
print(f"p-value: {p_value:.3f}")
print(f"Reject at 5%: {reject}")

p-value: 0.299
Reject at 5%: False


**Interpretation.** The fixed-PD test does not reject. Tiny counts make the chi-square inference unreliable.

### 13. Spiegelhalter test

The Spiegelhalter test compares squared forecast errors with their expectation under calibrated PDs. It tests one weighted calibration moment, so opposing errors can offset each other.

[Method reference and sources](../reference/spiegelhalter_test.md).

In [14]:
data = pd.DataFrame({'grade': ['A'] * 4 + ['B'] * 4, 'outcome': [0, 0, 1, 1] * 2, 'pd': [.2] * 4 + [.6] * 4})
result = m.spiegelhalter_test(data, 'grade', 'outcome', 'pd')
z_statistic, reject = result
print(f"z statistic: {z_statistic:.3f}")
print(f"Reject at 5%: {reject}")

z statistic: 1.543
Reject at 5%: False


**Interpretation.** The statistic is about 1.54 and does not reject at 5% under the two-sided convention.

### 14. Brier score

The Brier score is the mean squared difference between each predicted PD and its observed 0/1 default outcome. Lower scores indicate smaller probability forecast errors on the same observations.

[Method reference and sources](../reference/brier_score.md).

In [15]:
data = pd.DataFrame({'grade': ['A'] * 4 + ['B'] * 4, 'outcome': [0, 0, 1, 1] * 2, 'pd': [.2] * 4 + [.6] * 4})
result = m.brier_score(data, 'grade', 'outcome', 'pd')
print(f"Brier score: {result:.3f}")

Brier score: 0.300


**Interpretation.** Mean squared probability error is 0.30. Compare models on the same observations and event definition.

### 15. Redelmeier test

This comparison uses paired Brier losses from two sets of PD forecasts. Its null model places each true PD at the midpoint of the two forecasts and treats obligors as independent; a positive z statistic favours the second forecast.

[Method reference and sources](../reference/redelmeier_test.md).

In [16]:
data = pd.DataFrame({'y': [0, 1, 1, 0], 'p1': [.1, .4, .7, .3], 'p2': [.2, .6, .6, .1]})
result = m.redelmeier_test(data, default_flag='y', first_pd='p1', second_pd='p2')
z_statistic, p_value = result
print(f"z statistic: {z_statistic:.3f}")
print(f"p-value: {p_value:.3f}")

z statistic: 0.637
p-value: 0.524


**Interpretation.** First-forecast total squared loss exceeds the second by 0.18. Swapping forecasts reverses z and preserves the two-sided p-value.

### 16. Normal test for annual default rates

The normal test assesses whether realised portfolio default rates exceed predicted rates on average across years. It uses the mean annual error and its sample variance, with a normal reference distribution.

[Method reference and sources](../reference/normal_test.md).

In [17]:
result = m.normal_test([.1, .1, .1, .1], [.1, .2, .3, .4])
result.rename(columns={"estimate": "Mean PD error", "t_stat": "z statistic",
                       "p_value": "p-value", "outcome": "Reject at 5%"}).round(3)

,Mean PD error,z statistic,p-value,Reject at 5%
0,0.15,2.324,0.01,True


**Interpretation.** Errors 0, 0.1, 0.2, 0.3 give z about 2.324 and p about 0.010. Four years illustrate arithmetic, not a strong asymptotic basis.

## Association

### 17. Kendall's tau

Kendall's tau measures agreement between two rankings by comparing concordant and discordant pairs. The default tau-b convention accounts for ties in both variables.

[Method reference and sources](../reference/kendall_tau.md).

In [18]:
result = m.kendall_tau([1, 2, 3, 4], [1, 2, 4, 8])
tau, p_value = result
print(f"Kendall's tau: {tau:.3f}")
print(f"p-value: {p_value:.3f}")

Kendall's tau: 1.000
p-value: 0.083


**Interpretation.** All pairs concord: tau=1. The exact two-sided p-value for four distinct observations is 2/4!=1/12.

### 18. Somers' D

Somers' D measures how well the outcome supports distinctions made by the predictor. Rows of the example table represent ordered predicted grades and columns represent ordered realised grades.

[Method reference and sources](../reference/somersd.md).

In [19]:
result = m.somersd([[3, 1], [1, 3]])
print(f"Somers' D: {result.statistic:.3f}")
print(f"p-value: {result.pvalue:.3f}")

Somers' D: 0.500
p-value: 0.102


**Interpretation.** The association conditional on predicted grade is 0.500, with p = 0.102. The conditioning direction matters: predicted grades form the rows and realised grades form the columns.

### 19. Spearman's rho

Spearman's rho measures association between ranks. The example compares two variables whose orderings agree even though their numerical relationship is curved.

[Method reference and sources](../reference/spearman_correlation.md).

In [20]:
result = m.spearman_correlation([1, 2, 3, 4], [1, 2, 4, 8])
print(f"Spearman's rho: {result.statistic:.3f}")

Spearman's rho: 1.000


**Interpretation.** The relationship is strictly increasing: rank correlation equals 1 despite a nonlinear scale.

### 20. Pearson's r

Pearson's r measures linear association using numerical values. It uses the same observations as the Spearman example, making the difference between rank agreement and linear association visible.

[Method reference and sources](../reference/pearson_correlation.md).

In [21]:
result = m.pearson_correlation([1, 2, 3, 4], [1, 2, 4, 8])
print(f"Pearson's r: {result.statistic:.3f}")
print(f"p-value: {result.pvalue:.3f}")

Pearson's r: 0.959
p-value: 0.041


**Interpretation.** Pearson r is about 0.959, below Spearman r=1: monotone does not imply exactly linear.

## Stability

### 21. Herfindahl index

The Herfindahl index measures concentration across rating grades by adding their squared portfolio shares. Meliora also returns the coefficient of variation of those shares.

[Method reference and sources](../reference/herfindahl_test.md).

In [22]:
data = pd.DataFrame({'grade': ['A'] * 4 + ['B'] * 2})
result = m.herfindahl_test(data, 'grade')
coefficient_of_variation, hhi = result
print(f"Coefficient of variation: {coefficient_of_variation:.3f}")
print(f"Herfindahl index: {hhi:.3f}")

Coefficient of variation: 0.333


Herfindahl index: 0.556


**Interpretation.** HHI=5/9 exceeds the two-grade uniform baseline of 1/2. Compare on the same grade universe.

### 22. Multiple-period Herfindahl test

This method compares current grade concentration with a fixed development benchmark using the ECB statistic. The first table shows grade counts; the second shows the portfolio-level concentration comparison.

[Method reference and sources](../reference/herfindahl_multiple_period_test.md).

In [23]:
initial = pd.DataFrame({'grade': ['A'] * 4 + ['B'] * 2})
current = pd.DataFrame({'grade': ['A'] * 5 + ['B']})
result = m.herfindahl_multiple_period_test(initial, current, 'grade')
counts = result.drop(index="total")[["N_initial", "N_current"]]
comparison = result.loc[["total"], ["h_initial", "h_current", "z_stat", "p_value", "reject"]]
display(counts, comparison.round(3))

,N_initial,N_current
grade,,
A,4,5
B,2,1


,h_initial,h_current,z_stat,p_value,reject
grade,,,,,
total,0.556,0.722,0.514,0.303,False


**Interpretation.** The HHI rises from 0.556 to 0.722 as grade A grows from four of six accounts to five of six. The comparison gives z = 0.514 and p = 0.303, so the 5% rule is not triggered.

### 23. Population stability index

The population stability index compares grade or bin proportions between two samples. The output shows the two distributions and each bin's contribution to the total PSI.

[Method reference and sources](../reference/population_stability_index.md).

In [24]:
data = pd.DataFrame({'period': ['old'] * 4 + ['new'] * 4, 'bin': ['A', 'A', 'A', 'B', 'A', 'B', 'B', 'B']})
result = m.population_stability_index(data, 'period', 'bin', expected='old', actual='new', smoothing=0)
details, psi = result
print(f"Population stability index: {psi:.3f}")
details.round(3)

Population stability index: 1.099


,expected,actual,PSI
bin,,,
A,0.75,0.25,0.549
B,0.25,0.75,0.549


**Interpretation.** Shares change from (.75,.25) to (.25,.75), giving PSI=log(3). Significance requires a separate sampling model.

### 24. Migration bandwidth

Migration bandwidth measures how far migrating accounts move relative to the distances available on the grade scale. Grades here run from lower to higher risk, so movements above the diagonal are downgrades and those below it are upgrades.

[Method reference and sources](../reference/migration_matrices_statistics.md).

In [25]:
data = pd.DataFrame({'start': [1] * 4 + [2] * 4 + [3] * 4, 'end': [1, 1, 2, 3, 1, 2, 2, 3, 1, 2, 3, 3]})
result = m.migration_matrices_statistics(data, 'start', 'end', rating_order=[1, 2, 3])
downgrade_bandwidth, upgrade_bandwidth = result
print(f"Downgrade bandwidth: {downgrade_bandwidth:.3f}")
print(f"Upgrade bandwidth: {upgrade_bandwidth:.3f}")

Downgrade bandwidth: 0.800
Upgrade bandwidth: 0.800


**Interpretation.** Each side has distance-weighted count 4 and maximum-distance-weighted count 5, giving 0.8.

### 25. Adjacent-cell shape checks

These checks compare adjacent destination probabilities within one migration matrix. For each off-diagonal destination, the comparison uses the neighbouring probability one step closer to the diagonal. Diagonal cells are marked with a dash because the comparisons apply to off-diagonal cells.

[Method reference and sources](../reference/migration_matrix_stability.md).

In [26]:
data = pd.DataFrame({'start': [1] * 4 + [2] * 4 + [3] * 4, 'end': [1, 1, 2, 3, 1, 2, 2, 3, 1, 2, 3, 3]})
result = m.migration_matrix_stability(data, 'start', 'end', rating_order=[1, 2, 3])
z_scores, normal_cdfs = result
display(z_scores.style.format("{:.3f}", na_rep="\u2014").set_caption("z statistics"))
display(normal_cdfs.style.format("{:.3f}", na_rep="\u2014").set_caption("Normal CDFs"))

end,1,2,3
start,,,
1,—,0.603,0.000
2,0.603,—,0.603
3,0.000,0.603,—


end,1,2,3
start,,,
1,—,0.727,0.500
2,0.727,—,0.727
3,0.500,0.727,—


**Interpretation.** Cell (1,2) compares 2/4 on the diagonal against 1/4 nearby. Cell (1,3) has equal adjacent probabilities, giving CDF 0.5.

## LGD validation

### 26. LGD t-test

The LGD t-test assesses whether mean realised LGD exceeds mean predicted LGD. This example performs the paired comparison separately for two segments, giving each facility equal weight.

[Method reference and sources](../reference/lgd_t_test.md).

In [27]:
data = pd.DataFrame({'ead': [100, 200, 100, 100], 'predicted': [.2, .4, .6, .8], 'realised': [.1, .5, .4, .9], 'segment': ['A', 'A', 'B', 'B']})
result = m.lgd_t_test(data, 'realised', 'predicted', level='segment', segment_col='segment')
result.set_index("segment")[["pred_lgd_mean", "realised_lgd_mean", "t_stat", "p_value"]].round(3)

,pred_lgd_mean,realised_lgd_mean,t_stat,p_value
segment,,,,
A,0.3,0.30,-0.000,0.500
B,0.7,0.65,-0.333,0.602


**Interpretation.** Segment A has zero average error and p=0.5. Segment B has negative mean error, opposite the underestimation alternative.

### 27. ELBE t-test

The ELBE t-test compares realised LGD with the expected loss best estimate. Its two-sided alternative allows a difference in either direction, using paired facility-level errors.

[Method reference and sources](../reference/elbe_t_test.md).

In [28]:
data = pd.DataFrame({'ead': [100, 200, 100, 100], 'predicted': [.2, .4, .6, .8], 'realised': [.1, .5, .4, .9], 'segment': ['A', 'A', 'B', 'B']})
result = m.elbe_t_test(data, 'realised', 'predicted')
result.round(3)

,facilities,lgd_mean,elbe_mean,t_stat,p_value
0,4,0.475,0.5,-0.333,0.761


**Interpretation.** Realised mean LGD is 0.475 versus ELBE 0.5. This equal-facility test differs from an exposure-weighted score.

### 28. Loss shortfall

Loss shortfall compares predicted and realised monetary losses as a fraction of realised loss. A positive value indicates aggregate underestimation and a negative value indicates overestimation.

[Method reference and sources](../reference/loss_shortfall.md).

In [29]:
data = pd.DataFrame({'ead': [100, 200, 100, 100], 'predicted': [.2, .4, .6, .8], 'realised': [.1, .5, .4, .9], 'segment': ['A', 'A', 'B', 'B']})
result = m.loss_shortfall(data, 'ead', 'predicted', 'realised')
print(f"Loss shortfall: {result:.3f}")

Loss shortfall: 0.000


**Interpretation.** Predicted and realised monetary losses both total 240, giving a shortfall of 0.000. The following mean-absolute-error example shows the individual errors behind these equal totals.

### 29. Exposure-weighted mean absolute error

Exposure-weighted mean absolute error measures the size of individual LGD errors. Absolute errors are weighted by exposure, so overestimates and underestimates do not cancel each other.

[Method reference and sources](../reference/mean_absolute_deviation.md).

In [30]:
data = pd.DataFrame({'ead': [100, 200, 100, 100], 'predicted': [.2, .4, .6, .8], 'realised': [.1, .5, .4, .9], 'segment': ['A', 'A', 'B', 'B']})
result = m.mean_absolute_deviation(data, 'ead', 'predicted', 'realised')
print(f"Exposure-weighted mean absolute error: {result:.3f}")

Exposure-weighted mean absolute error: 0.120


**Interpretation.** The exposure-weighted absolute error is 0.120, or 12 LGD percentage points. The preceding loss-shortfall example has the same predicted and realised total loss, while this measure reveals the individual forecast errors.